In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("ANTHROPIC_API_KEY")

if token:
    masked = f"{token[:4]}...{token[-4:]}" if len(token) > 8 else "****"
    print(f"ANTHROPIC_API_KEY loaded ({len(token)} characters): {masked}")
else:
    print("ANTHROPIC_API_KEY not found. Check that .env exists in this directory, "
          "the key is spelled exactly 'ANTHROPIC_API_KEY', and load_dotenv() ran without error.")

ANTHROPIC_API_KEY loaded (108 characters): sk-a...4wAA


In [2]:
"""
Few-shot relevance classification using Claude Sonnet (Anthropic API).

Same crash-safe / resumable design as the other classify_with_* scripts.

Requires: pip install anthropic
Requires environment variable: ANTHROPIC_API_KEY
"""

import json
import os
import re
import time
import pandas as pd
import anthropic

# ---- Config ----
MODEL_NAME = "claude-sonnet-5"  # current Sonnet model string as of this project;
                                  # check https://docs.claude.com for the latest if this errors
PROMPT_FILE = "gpt_classification_prompt.txt"
INPUT_FILE = "merged_shuffled.xlsx"
OUTPUT_JSONL = "classification_results_sonnet.jsonl"
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5


def load_system_prompt(path: str) -> str:
    with open(path) as f:
        return f.read()


def load_input_papers(path: str) -> pd.DataFrame:
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")


def already_processed_ids(output_path: str) -> set:
    if not os.path.exists(output_path):
        return set()
    ids = set()
    with open(output_path) as f:
        for line in f:
            try:
                ids.add(json.loads(line)["wos_id"])
            except (json.JSONDecodeError, KeyError):
                continue
    return ids


def extract_json(text: str) -> dict | None:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def build_user_message(title: str, abstract: str) -> str:
    return f"Title: {title}\nAbstract: {abstract}"


def extract_text_from_response(response) -> str:
    """Find the actual text block in the response, rather than assuming
    content[0] is text. With extended thinking enabled, content[0] is often
    a ThinkingBlock (has .thinking, not .text), with the real answer in a
    later block. Also handles the case where content is empty entirely."""
    if not response.content:
        # Empty content can mean several different things -- surface the
        # actual stop_reason and token usage so we can tell them apart
        # (safety refusal vs. thinking consuming the whole token budget vs.
        # something else) instead of guessing.
        stop_reason = getattr(response, "stop_reason", "unknown")
        usage = getattr(response, "usage", None)
        usage_str = (
            f"input={usage.input_tokens}, output={usage.output_tokens}"
            if usage else "unavailable"
        )
        raise ValueError(
            f"Response content is empty. stop_reason={stop_reason!r}, "
            f"token usage: {usage_str}"
        )
    for block in response.content:
        if getattr(block, "type", None) == "text":
            return block.text
    # No text block found among any of the returned blocks (e.g. only
    # thinking blocks, no final answer) -- surface this clearly rather than
    # crashing on an index/attribute error.
    block_types = [getattr(b, "type", type(b).__name__) for b in response.content]
    stop_reason = getattr(response, "stop_reason", "unknown")
    raise ValueError(
        f"No text block found in response. Block types present: {block_types}, "
        f"stop_reason={stop_reason!r}"
    )


def call_with_retries(client, system_prompt, user_message):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=1500,  # raised from 300 -- this model uses extended
                                   # thinking by default, and thinking tokens
                                   # count against this budget before the
                                   # actual JSON answer is produced
                system=system_prompt,
                messages=[{"role": "user", "content": user_message}],
            )
            return extract_text_from_response(response)
        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt} failed: {e}. Retrying in {RETRY_DELAY_SECONDS}s...")
            time.sleep(RETRY_DELAY_SECONDS)
    raise last_error


def main():
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    system_prompt = load_system_prompt(PROMPT_FILE)
    papers = load_input_papers(INPUT_FILE)
    done_ids = already_processed_ids(OUTPUT_JSONL)
    print(f"Total papers: {len(papers)} | Already processed: {len(done_ids)}")

    n_ok, n_parse_failed, n_api_failed = 0, 0, 0
    flagged = []

    with open(OUTPUT_JSONL, "a") as out_f:
        for row in papers.itertuples():
            wos_id = row.wos_id
            if wos_id in done_ids:
                continue

            user_message = build_user_message(row.title, row.abstract)
            try:
                generated = call_with_retries(client, system_prompt, user_message)
            except Exception as e:
                print(f"  FAILED after retries for {wos_id}: {e}")
                n_api_failed += 1
                flagged.append(wos_id)
                continue

            parsed = extract_json(generated)
            result = {"wos_id": wos_id, "title": row.title, "raw_model_output": generated}
            if parsed is not None and "label" in parsed:
                result.update(parsed)
                n_ok += 1
            else:
                result.update({"label": None, "trap_reason": None, "reason": None, "parse_failed": True})
                n_parse_failed += 1
                flagged.append(wos_id)

            out_f.write(json.dumps(result) + "\n")
            out_f.flush()

    print(f"\nDone. OK: {n_ok} | Parse failed: {n_parse_failed} | API failed: {n_api_failed}")
    if flagged:
        print(f"Flagged wos_ids: {flagged[:20]}{' ...' if len(flagged) > 20 else ''}")
    print(f"Results saved to {OUTPUT_JSONL}")


if __name__ == "__main__":
    main()

Total papers: 433 | Already processed: 0
  Attempt 1 failed: Response content is empty. stop_reason='refusal', token usage: input=4498, output=0. Retrying in 5s...
  Attempt 2 failed: Response content is empty. stop_reason='refusal', token usage: input=4498, output=0. Retrying in 5s...
  Attempt 3 failed: Response content is empty. stop_reason='refusal', token usage: input=4498, output=0. Retrying in 5s...
  FAILED after retries for syn116: Response content is empty. stop_reason='refusal', token usage: input=4498, output=0
  Attempt 1 failed: Response content is empty. stop_reason='refusal', token usage: input=4454, output=0. Retrying in 5s...
  Attempt 2 failed: Response content is empty. stop_reason='refusal', token usage: input=4454, output=0. Retrying in 5s...
  Attempt 3 failed: Response content is empty. stop_reason='refusal', token usage: input=4454, output=0. Retrying in 5s...
  FAILED after retries for syn24: Response content is empty. stop_reason='refusal', token usage: input

# convert to xlsx

In [3]:
"""
Convert a classification results .jsonl file (from classify_with_open_llm.py,
or your dpo_pairs.jsonl / train_pairs.jsonl / validation_pairs.jsonl) into an
.xlsx file for easy review in Excel.

Usage: edit INPUT_PATH and OUTPUT_PATH below, then run.
"""

import json
import pandas as pd

INPUT_PATH = "classification_results_sonnet.jsonl"
OUTPUT_PATH = "LLM-FULL/classification_results_sonnet.xlsx"


def main():
    rows = []
    with open(INPUT_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    df = pd.DataFrame(rows)
    df.to_excel(OUTPUT_PATH, index=False)
    print(f"Converted {len(rows)} rows from {INPUT_PATH} -> {OUTPUT_PATH}")
    print(f"Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

Converted 429 rows from classification_results_sonnet.jsonl -> LLM-FULL/classification_results_sonnet.xlsx
Columns: ['wos_id', 'title', 'raw_model_output', 'label', 'trap_reason', 'reason']


In [4]:
"""
Re-parse rows that failed JSON extraction, using a hardened parser that
handles two common, harmless LLM formatting quirks:
  1. Literal newline characters inside a JSON string value (should be
     escaped as \\n, but the model sometimes just line-wraps the text).
  2. Curly/smart quotes (" " ' ') used instead of straight ASCII quotes.

This does NOT call the model again -- it re-parses raw_model_output, which
is already saved from the original run. Only rows where parsing genuinely
fixes something are updated; rows still unparseable (e.g. the model
abandoned JSON entirely) stay flagged for manual review.

Usage: edit INPUT_PATH and OUTPUT_PATH, then run.
"""

import json
import re
import pandas as pd

INPUT_PATH = "LLM-FULL/classification_results_sonnet.xlsx"
OUTPUT_PATH = "LLM-Result/classification_results_sonnet_reparsed.xlsx"


def hardened_extract_json(text: str) -> dict | None:
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    candidate = match.group(0)

    # Fix 1: normalize smart/curly quotes to straight ASCII quotes.
    candidate = (
        candidate.replace("\u201c", '"').replace("\u201d", '"')
        .replace("\u2018", "'").replace("\u2019", "'")
    )

    # Try parsing as-is first.
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass

    # Fix 2: escape literal newlines/tabs that fall INSIDE string values.
    # A simple, safe approach for this use case: replace raw newlines/tabs
    # with a space, since these failures are all mid-sentence line wraps,
    # not intentional formatting.
    candidate_fixed = candidate.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    candidate_fixed = re.sub(r"\s+", " ", candidate_fixed)
    try:
        return json.loads(candidate_fixed)
    except json.JSONDecodeError:
        return None


def main():
    df = pd.read_excel(INPUT_PATH) if INPUT_PATH.endswith((".xlsx", ".xls")) else pd.read_csv(INPUT_PATH)

    if "parse_failed" not in df.columns:
        print("No 'parse_failed' column found -- nothing to re-parse.")
        return

    failed_mask = df["parse_failed"] == True
    n_failed_before = failed_mask.sum()
    print(f"Rows previously flagged as parse_failed: {n_failed_before}")

    # Force flexible dtype on columns we're about to write mixed values into,
    # since pandas can otherwise reject e.g. writing a bool into a float64
    # column that only had True/NaN in it originally.
    for col in ["label", "trap_reason", "reason", "parse_failed"]:
        if col in df.columns:
            df[col] = df[col].astype(object)

    n_recovered = 0
    still_failed = []

    for idx in df[failed_mask].index:
        raw = df.at[idx, "raw_model_output"]
        parsed = hardened_extract_json(raw)
        if parsed is not None and "label" in parsed:
            df.at[idx, "label"] = parsed.get("label")
            df.at[idx, "trap_reason"] = parsed.get("trap_reason")
            df.at[idx, "reason"] = parsed.get("reason")
            df.at[idx, "parse_failed"] = False
            n_recovered += 1
        else:
            still_failed.append(df.at[idx, "wos_id"])

    df.to_excel(OUTPUT_PATH, index=False)

    print(f"Recovered: {n_recovered}")
    print(f"Still failed (genuine issue, needs manual review or a rerun "
          f"for just this row): {len(still_failed)}")
    if still_failed:
        print(f"  -> {still_failed}")
    print(f"Saved to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

No 'parse_failed' column found -- nothing to re-parse.
